# Collab For Running Compatible API Of Qwen3:5-4B  

---

**IMPORTANT NOTE:** This notebook was ran on google collab

**FILES:** All files come from the mounted drive

In [1]:
!nvidia-smi

Tue Oct 27 12:42:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3.5-4B"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Ответь кратко по-русски: что такое голоморфная функция?"
            }
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

answer = outputs[0][inputs["input_ids"].shape[-1]:]
print(processor.decode(answer, skip_special_tokens=True))

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What is 2 + 2? Answer briefly."}
        ],
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(repr(prompt[-300:]))

'<|im_start|>user\nWhat is 2 + 2? Answer briefly.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [ ]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

print(processor.decode(new_tokens, skip_special_tokens=False))

4<|im_end|>
<|endoftext|>


In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "In complex analysis, is f(z) = z^2 holomorphic everywhere "
                    "in the complex plane? Answer briefly and explain why."
                )
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

response = processor.decode(new_tokens, skip_special_tokens=True)
print("QUESTION:", messages[0]["content"][0]["text"])
print("\nQWEN RESPONSE:\n")
print(response)

QUESTION: In complex analysis, is f(z) = z^2 holomorphic everywhere in the complex plane? Answer briefly and explain why.

QWEN RESPONSE:

**Yes**, the function $f(z) = z^2$ is holomorphic everywhere in the complex plane.

**Explanation:**
A function is holomorphic at a point if it is complex differentiable in a neighborhood of that point. For $f(z) = z^2$, the derivative exists for all $z \in \mathbb{C}$ and is given by:
$$f'(z) = 2z$$
Since the derivative $2z$ is defined and finite for every complex number $z$, the function is differentiable everywhere, making it an **entire function**. Additionally, $z^2$ is a polynomial, and all polynomials are holomorphic on the entire complex plane.



In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Evidence-RAG")

print("Project files:")

if PROJECT.exists():
    for path in sorted(PROJECT.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT))
else:
    print(f"Directory {PROJECT} does not exist. Please check the path.")


In [6]:
print("Model:", model.config._name_or_path)
print("Model type:", model.config.model_type)
print("EOS token ID:", processor.tokenizer.eos_token_id)
print("Chat template available:", processor.chat_template is not None)
print("\nGeneration defaults:")
print(model.generation_config)

Model: Qwen/Qwen3.5-4B
Model type: qwen3_5
EOS token ID: 248046
Chat template available: True

Generation defaults:
GenerationConfig {
  "eos_token_id": 248044,
  "use_cache": true
}



In [7]:
%cd /content/drive/MyDrive/Evidence-RAG

/content/drive/MyDrive/Evidence-RAG


In [8]:
from pathlib import Path

for name in [
    "build_index.py",
    "config.json",
    "data/dev_questions.jsonl",
    "index/bm25.pkl",
    "index/documents.jsonl",
]:
    p = Path(name)
    print("OK     " if p.exists() else "MISSING", name)

OK      build_index.py
OK      config.json
OK      data/dev_questions.jsonl
OK      index/bm25.pkl
OK      index/documents.jsonl


In [ ]:
#%pip install rank-bm25

In [ ]:
import json
import pickle

# Load documents
with open(PROJECT / "index/documents.jsonl", encoding="utf-8") as f:
    documents = [json.loads(line) for line in f if line.strip()]

# Inspect the index format
with open(PROJECT / "index/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

print("Documents loaded:", len(documents))
print("Index type:", type(bm25).__name__)
print("\nFirst document fields:", list(documents[1].keys()))
print("First document preview:", str(documents[1])[:500])

Documents loaded: 968
Index type: BM25Okapi

First document fields: ['block_id', 'page', 'section', 'text']
First document preview: {'block_id': 'b0000002', 'page': 2, 'section': '1.1. Алгебра комплексных чисел', 'text': '1.1. Алгебра комплексных чисел. С точки зрения разрешимости алгеб-\n\nраического уравнения x2 + 1 = 0 появляется необходимость расширения поля вещественных чисел. Пусть i новое число (мнимая единица), которое удовлетворяет условию i2 = i · i = −1. Если производить операции сложения и умножения над вещественными числами и мнимой единицей, то мы естественным образом приходим к записи комплексных чисел z = x +


In [ ]:
#%pip -q install fastapi uvicorn

In [12]:
import time
import threading
import torch
import uvicorn
from fastapi import FastAPI, Request, HTTPException

app = FastAPI()
server_config = uvicorn.Config(
    app, host="127.0.0.1", port=8000, log_level="warning"
)
server = uvicorn.Server(server_config)

@app.get("/health")
def health():
    return {"status": "ok", "model": "Qwen/Qwen3.5-4B"}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()

    messages = body.get("messages")
    if not isinstance(messages, list) or not messages:
        raise HTTPException(400, "messages must be a non-empty list")

    max_tokens = int(body.get("max_tokens", 700))
    if not 1 <= max_tokens <= 4096:
        raise HTTPException(400, "max_tokens must be between 1 and 4096")

    try:
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(model.device)

        prompt_tokens = int(inputs["input_ids"].shape[-1])

        generation_args = {
            "max_new_tokens": max_tokens,
            "do_sample": float(body.get("temperature", 0.0)) > 0,
        }
        if generation_args["do_sample"]:
            generation_args["temperature"] = float(body["temperature"])
        if "top_p" in body:
            generation_args["top_p"] = float(body["top_p"])

        start = time.perf_counter()
        with torch.inference_mode():
            output_ids = model.generate(**inputs, **generation_args)
        elapsed = time.perf_counter() - start

        new_ids = output_ids[0, prompt_tokens:]
        completion_tokens = int(new_ids.shape[-1])
        text = processor.decode(new_ids, skip_special_tokens=True)

        return {
            "id": "local-qwen-completion",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": body.get("model", "Qwen/Qwen3.5-4B"),
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "length" if completion_tokens >= max_tokens else "stop",
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
            "local_generation_seconds": round(elapsed, 3),
        }

    except Exception as exc:
        raise HTTPException(500, f"Local model generation failed: {exc}") from exc

# Start the server in the background, without loading the model again.
if "api_thread" not in globals() or not api_thread.is_alive():
    server = uvicorn.Server(
        uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    )
    api_thread = threading.Thread(target=server.run, daemon=True)
    api_thread.start()

# Wait for the server to become available.
import requests
for _ in range(30):
    try:
        response = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if response.ok:
            print(response.json())
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Local Qwen API did not start.")


{'status': 'ok', 'model': 'Qwen/Qwen3.5-4B'}


In [13]:
import requests, json

test_payload = {
    "model": "Qwen/Qwen3.5-4B",
    "messages": [
        {"role": "user", "content": "Return exactly this JSON: {\"ok\": true}"}
    ],
    "temperature": 0.0,
    "max_tokens": 80
}

r = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json=test_payload,
    timeout=180,
)
print("HTTP:", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))
r.raise_for_status()


HTTP: 200
{
  "id": "local-qwen-completion",
  "object": "chat.completion",
  "created": 1793112370,
  "model": "Qwen/Qwen3.5-4B",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\"ok\": true}\n"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 22,
    "completion_tokens": 8,
    "total_tokens": 30
  },
  "local_generation_seconds": 1.646
}


In [14]:
import os

os.environ["LLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
os.environ["LLM_MODEL"] = "Qwen/Qwen3.5-4B"
os.environ["SUBMISSION_ID"] = "local-dev"
# LLM_API_KEY is optional for the local adapter.
os.environ.pop("LLM_API_KEY", None)

print("LLM endpoint:", os.environ["LLM_BASE_URL"])
print("Model:", os.environ["LLM_MODEL"])


LLM endpoint: http://127.0.0.1:8000/v1
Model: Qwen/Qwen3.5-4B


In [ ]:
import json
from pathlib import Path

questions_path = PROJECT / "data" / "dev_questions.jsonl"
one_question_path = PROJECT / "data" / "one_question.jsonl"

with open(questions_path, "r", encoding="utf-8") as f:
    first_question = next(line for line in f if line.strip())

with open(one_question_path, "w", encoding="utf-8") as f:
    f.write(first_question.rstrip() + "\n")

print("Test question:")
print(json.dumps(json.loads(first_question), ensure_ascii=False, indent=2))


Test question:
{
  "id": "q_001",
  "question": "Пусть $f=u+iv$ определена в окрестности $z_0$. Какие утверждения о комплексной дифференцируемости в точке $z_0$ верны?",
  "options": [
    {
      "id": "A",
      "text": "Существования первых частных производных u и v в точке и выполнения условий Коши—Римана в ней всегда достаточно."
    },
    {
      "id": "B",
      "text": "Вещественная дифференцируемость u и v вместе с условиями Коши—Римана необходима и достаточна."
    },
    {
      "id": "C",
      "text": "Комплексная дифференцируемость в точке влечёт вещественную дифференцируемость в ней."
    },
    {
      "id": "D",
      "text": "Для существования комплексной производной в одной точке обязательно требуется голоморфность в некоторой её окрестности."
    }
  ]
}


In [16]:
CONFIG_FILE_PATH = PROJECT / "runtime.json"


def update_experiment_name(new_name: str, file_path: Path = CONFIG_FILE_PATH):
    """Loads the JSON configuration, updates the experiment name, and saves it back."""
    if file_path.exists():
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        print("file not found!")

    data["experiment"]["name"] = new_name

    file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"Successfully updated experiment name to: '{new_name}'")


In [15]:
from google.colab import files

In [ ]:
update_experiment_name("final")

Successfully updated experiment name to: 'final'


In [ ]:
!python predict.py --experiment-name final \
  --index ./index --questions ./data/dev_questions.jsonl \
  --output ./dev_answers.jsonl --trace ./dev_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: final ===
  answers → dev_answers_final.jsonl
  trace   → dev_trace_final.jsonl
[1/15] q_001: ['B', 'C']
[2/15] q_002: Отказ
[3/15] q_003: ['B', 'C']
[4/15] q_004: ['B', 'C']
[5/15] q_005: ['A', 'B']
[6/15] q_006: ['A', 'D']
[7/15] q_007: Отказ
[8/15] q_008: ['B']
[9/15] q_009: ['A', 'C']
[10/15] q_010: ['A', 'B']
[11/15] q_011: ['A', 'C']
[12/15] q_012: ['A', 'C']
[13/15] q_013: Отказ
[14/15] q_014: Отказ
[15/15] q_015: Отказ
Done. Wrote dev_answers_final.jsonl and dev_trace_final.jsonl


In [ ]:
files.download('dev_answers_final.jsonl')
files.download('dev_trace_final.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
update_experiment_name("no_context")

In [ ]:
!python predict.py --experiment-name no_context \
  --index ./index --questions ./data/dev_questions.jsonl \
  --output ./dev_answers.jsonl --trace ./dev_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: no_context ===
  answers → dev_answers_no_context.jsonl
  trace   → dev_trace_no_context.jsonl
[1/15] q_001: ['B', 'C']
[2/15] q_002: ['A', 'D']
[3/15] q_003: ['B', 'C']
[4/15] q_004: ['B', 'C']
[5/15] q_005: ['B', 'C']
[6/15] q_006: ['A', 'D']
[7/15] q_007: ['B']
[8/15] q_008: ['B']
[9/15] q_009: ['A', 'C']
[10/15] q_010: ['A', 'B', 'C']
[11/15] q_011: ['A', 'C']
[12/15] q_012: ['B', 'C']
[13/15] q_013: Отказ
[14/15] q_014: Отказ
[15/15] q_015: Отказ
Done. Wrote dev_answers_no_context.jsonl and dev_trace_no_context.jsonl


In [ ]:
files.download('dev_answers_no_context.jsonl')
files.download('dev_trace_no_context.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
update_experiment_name("rag_single")

Successfully updated experiment name to: 'rag_single'


In [ ]:
!python predict.py --experiment-name rag_single \
  --index ./index --questions ./data/dev_questions.jsonl \
  --output ./dev_answers.jsonl --trace ./dev_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: rag_single ===
  answers → dev_answers_rag_single.jsonl
  trace   → dev_trace_rag_single.jsonl
[1/15] q_001: ['B', 'C']
[2/15] q_002: Отказ
[3/15] q_003: ['B', 'C']
[4/15] q_004: ['B', 'C']
[5/15] q_005: ['A', 'B']
[6/15] q_006: ['A', 'D']
[7/15] q_007: Отказ
[8/15] q_008: ['B']
[9/15] q_009: ['A', 'C']
[10/15] q_010: ['A', 'B']
[11/15] q_011: ['A', 'C']
[12/15] q_012: ['A', 'C']
[13/15] q_013: Отказ
[14/15] q_014: Отказ
[15/15] q_015: Отказ
Done. Wrote dev_answers_rag_single.jsonl and dev_trace_rag_single.jsonl


In [ ]:
files.download('dev_answers_rag_single.jsonl')
files.download('dev_trace_rag_single.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
update_experiment_name("ablation_no_rerank")

Successfully updated experiment name to: 'ablation_no_rerank'


In [ ]:
!python predict.py --experiment-name ablation_no_rerank \
  --index ./index --questions ./data/dev_questions.jsonl \
  --output ./dev_answers.jsonl --trace ./dev_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: ablation_no_rerank ===
  answers → dev_answers_ablation_no_rerank.jsonl
  trace   → dev_trace_ablation_no_rerank.jsonl
[1/15] q_001: ['B', 'C']
[2/15] q_002: ['A', 'D']
[3/15] q_003: ['B', 'C']
[4/15] q_004: ['B', 'C']
[5/15] q_005: Отказ
[6/15] q_006: ['A', 'D']
[7/15] q_007: Отказ
[8/15] q_008: ['A', 'B']
[9/15] q_009: ['A']
[10/15] q_010: ['A', 'B']
[11/15] q_011: Отказ
[12/15] q_012: ['A', 'C']
[13/15] q_013: Отказ
[14/15] q_014: Отказ
[15/15] q_015: Отказ
Done. Wrote dev_answers_ablation_no_rerank.jsonl and dev_trace_ablation_no_rerank.jsonl


In [ ]:
files.download('dev_answers_ablation_no_rerank.jsonl')
files.download('dev_trace_ablation_no_rerank.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
update_experiment_name("ablation_top1")

Successfully updated experiment name to: 'ablation_top1'


In [ ]:
!python predict.py --experiment-name ablation_top1 \
  --index ./index --questions ./data/dev_questions.jsonl \
  --output ./dev_answers.jsonl --trace ./dev_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: ablation_top1 ===
  answers → dev_answers_ablation_top1.jsonl
  trace   → dev_trace_ablation_top1.jsonl
[1/15] q_001: ['B', 'C']
[2/15] q_002: Отказ
[3/15] q_003: ['B', 'C']
[4/15] q_004: ['B', 'C']
[5/15] q_005: ['A', 'B']
[6/15] q_006: ['A', 'D']
[7/15] q_007: Отказ
[8/15] q_008: ['B']
[9/15] q_009: ['A', 'C']
[10/15] q_010: ['A', 'B']
[11/15] q_011: ['A', 'C']
[12/15] q_012: ['C']
[13/15] q_013: Отказ
[14/15] q_014: Отказ
[15/15] q_015: Отказ
Done. Wrote dev_answers_ablation_top1.jsonl and dev_trace_ablation_top1.jsonl


In [ ]:
files.download('dev_answers_ablation_top1.jsonl')
files.download('dev_trace_ablation_top1.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Run Final Model On Own Questions:

In [17]:
update_experiment_name("final")

Successfully updated experiment name to: 'final'


In [18]:
!python predict.py --experiment-name final \
  --index ./index --questions ./data/own_questions.jsonl \
  --output ./own_answers.jsonl --trace ./own_trace.jsonl \
  --config ./config.json --runtime ./runtime.json

=== Experiment: final ===
  answers → own_answers_final.jsonl
  trace   → own_trace_final.jsonl
[1/8] q_own_001: ['A', 'B', 'C']
[2/8] q_own_002: ['A', 'C']
[3/8] q_own_003: ['A', 'B', 'C']
[4/8] q_own_004: ['A', 'B', 'C']
[5/8] q_own_005: Отказ
[6/8] q_own_006: Отказ
[7/8] q_own_007: Отказ
[8/8] q_own_008: Отказ
Done. Wrote own_answers_final.jsonl and own_trace_final.jsonl


In [20]:
files.download('own_answers_final.jsonl')
files.download('own_trace_final.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import json
import statistics
from pathlib import Path

project = Path("PROJECT")
if not project.is_dir():
    project = Path(".")

answers_path = project / "own_answers_final.jsonl"
trace_path = project / "own_trace_final.jsonl"

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

answers = read_jsonl(answers_path)
traces = read_jsonl(trace_path)

if not traces:
    raise ValueError(f"No trace records found in {trace_path}")

input_tokens = [t.get("input_tokens", 0) for t in traces]
output_tokens = [t.get("output_tokens", 0) for t in traces]
llm_calls = [t.get("llm_calls", 0) for t in traces]
latencies = [t.get("elapsed_ms", 0) for t in traces]


costs = [
    t.get("input_tokens", 0)
    + 4 * t.get("output_tokens", 0)
    + 256 * t.get("llm_calls", 0)
    for t in traces
]

answer_ids = {str(a["id"]) for a in answers}
trace_ids = {str(t["id"]) for t in traces}

print("=== Own Test Set: Usage Metrics ===")
print(f"Answers:                 {len(answers)}")
print(f"Trace records:            {len(traces)}")
print(f"Questions missing trace:  {len(answer_ids - trace_ids)}")
print(f"Questions missing answer: {len(trace_ids - answer_ids)}")

print("\n--- Token and API Usage ---")
print(f"Total input tokens:       {sum(input_tokens):,}")
print(f"Average input tokens:     {statistics.mean(input_tokens):.1f}")
print(f"Total output tokens:      {sum(output_tokens):,}")
print(f"Average output tokens:    {statistics.mean(output_tokens):.1f}")
print(f"Total LLM calls:          {sum(llm_calls)}")
print(f"Average LLM calls:        {statistics.mean(llm_calls):.2f}")
print(f"Average cost:             {statistics.mean(costs):.1f}")
print(f"Total cost:               {sum(costs):,}")
print(f"Median latency:           {statistics.median(latencies):.0f} ms")
print(f"Average latency:          {statistics.mean(latencies):.0f} ms")

=== Own Test Set: Usage Metrics ===
Answers:                 8
Trace records:            8
Questions missing trace:  0
Questions missing answer: 0

--- Token and API Usage ---
Total input tokens:       9,894
Average input tokens:     1236.8
Total output tokens:      830
Average output tokens:    103.8
Total LLM calls:          8
Average LLM calls:        1.00
Average cost:             1907.8
Total cost:               15,262
Median latency:           11558 ms
Average latency:          13083 ms
